# Import Libraries

In [121]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Data

In [102]:
data = pd.read_csv("tmdb_5000_credits.csv")

# EDA

In [103]:
data.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [104]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [105]:
data.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

In [106]:
data.duplicated().sum()

np.int64(0)

# Data Preprocessing

In [107]:
# Converting the string representation of lists and 
# dictionaries to actual lists and dictionaries to be able to extract the required information
data['cast'] = data['cast'].apply(ast.literal_eval)
data['crew'] = data['crew'].apply(ast.literal_eval)

In [108]:
# Extracting the top 3 actor names from the cast list
def get_top_actors(cast):
    return [actor['name'] for actor in cast[:3]]

data['actors'] = data['cast'].apply(get_top_actors)


In [109]:
# Extracting the director's name from the crew list
def get_director(crew):
    for person in crew:
        if person['job'] == 'Director':
            return person['name']
    return None

data['director'] = data['crew'].apply(get_director)

In [110]:
data.isnull().sum()

movie_id     0
title        0
cast         0
crew         0
actors       0
director    30
dtype: int64

In [111]:
data.fillna('', inplace=True)

In [112]:
# Removing spaces from actor and director names to avoid issues during vectorization
data['actors'] = data['actors'].apply(lambda x: [i.replace(" ", "") for i in x])
data['director'] = data['director'].apply(lambda x: x.replace(" ", ""))


In [113]:
# Convert actor and director names to lowercase to avoid case sensitivity issues during vectorization
data['actors'] = data['actors'].apply(lambda x: [i.lower() for i in x])
data['director'] = data['director'].apply(lambda x: x.lower())

In [114]:
# Creating a combined tag column for recommendation
data['tags'] = data['actors'].apply(lambda x: " ".join(x)) + " " + data['director']

In [115]:
# Selecting only the required columns for the recommendation system
data = data.drop(["movie_id", "cast", "crew", "actors", "director"], axis=1)

In [127]:
data.isnull().sum()
data.duplicated().sum()

np.int64(0)

In [117]:
data.head()

,title,tags
0,Avatar,samworthington zoesaldana sigourneyweaver jame...
1,Pirates of the Caribbean: At World's End,johnnydepp orlandobloom keiraknightley gorever...
2,Spectre,danielcraig christophwaltz léaseydoux sammendes
3,The Dark Knight Rises,christianbale michaelcaine garyoldman christop...
4,John Carter,taylorkitsch lynncollins samanthamorton andrew...


In [ ]:
# Vectorizing the tags using TF-IDF vectorizer
tfidf = TfidfVectorizer()
vectors = tfidf.fit_transform(data['tags']).toarray()

# ML Model

In [123]:
similarity = cosine_similarity(vectors)

In [124]:
def recommend(movie):
    movie = movie.lower()
    index = data[data['title'].str.lower() == movie].index[0]
    distances = similarity[index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    for i in movies_list:
        print(data.iloc[i[0]].title)

In [125]:
recommend("Avatar")

Aliens
Clash of the Titans
Wrath of the Titans
Terminator Salvation
True Lies
